# Preparação de Dados

## 1 Byte pair encoding de palavras fora do voculário

Durante a aula vimos que um tokenizador baseado em Byte pair encoding (BPE) é capaz de lidar com palavras fora do vocabulário ao dividir uma palavra em "sub-palavras" que estejam presentes no vocabulário. Na pior das hipóteses a palavra pode ser quebrada em letras individuais.

O texto abaixo é um trecho tirado do primeiro parágrafo do livro "The Time Machine" (H. G. Wells, 1895). Use o Tiktoken (com encoding do gpt2) visto durante a aula para tokenizá-lo e verifique quais palavras não estão presentes no vocabulário e necessitaram ser quebradas em "sub-palavras". Mostre como ficou a divisão de cada uma das palavras originalmente fora do vocabulário após a tokenização.

Por exemplo, a palavra "luxurious":<br>
`luxurious -> ['lux', 'urious']`

In [57]:
time_machine_text = 'The Time Traveller was expounding a recondite matter to us. \
His grey eyes shone and twinkled, and his usually pale face was flushed and animated.'

In [58]:
import tiktoken

tokenizador_tiktoken = tiktoken.get_encoding("gpt2")
print("Texto original do exercicio:")
print(time_machine_text)

texto_time_machine_tokenizado_ids = tokenizador_tiktoken.encode(time_machine_text)
print("\nTexto após passar pelo tokenizador e ficar na forma de tokens ids:")
print(texto_time_machine_tokenizado_ids)
print(type(texto_time_machine_tokenizado_ids))
# posso tentar percorrer um loop for para mostrar palavra por palavra e achar as repartidas:
for token_id in texto_time_machine_tokenizado_ids:
    print(f"Token id: {token_id} -> {tokenizador_tiktoken.decode([token_id])}")


Texto original do exercicio:
The Time Traveller was expounding a recondite matter to us. His grey eyes shone and twinkled, and his usually pale face was flushed and animated.

Texto após passar pelo tokenizador e ficar na forma de tokens ids:
[464, 3862, 43662, 6051, 373, 1033, 9969, 257, 664, 623, 578, 2300, 284, 514, 13, 2399, 13791, 2951, 44193, 290, 665, 676, 992, 11, 290, 465, 3221, 14005, 1986, 373, 44869, 290, 15108, 13]
<class 'list'>
Token id: 464 -> The
Token id: 3862 ->  Time
Token id: 43662 ->  Trave
Token id: 6051 -> ller
Token id: 373 ->  was
Token id: 1033 ->  exp
Token id: 9969 -> ounding
Token id: 257 ->  a
Token id: 664 ->  rec
Token id: 623 -> ond
Token id: 578 -> ite
Token id: 2300 ->  matter
Token id: 284 ->  to
Token id: 514 ->  us
Token id: 13 -> .
Token id: 2399 ->  His
Token id: 13791 ->  grey
Token id: 2951 ->  eyes
Token id: 44193 ->  shone
Token id: 290 ->  and
Token id: 665 ->  tw
Token id: 676 -> ink
Token id: 992 -> led
Token id: 11 -> ,
Token id: 290 -> 

Ficou ruim de ver. Vou tentar voltar para lista desconvertendo em partes e não direto para não reconstruir o texto.

In [59]:
texto_time_machine_tokenizado_subwords = []
for token_id in texto_time_machine_tokenizado_ids:
    texto_time_machine_tokenizado_subwords.append(tokenizador_tiktoken.decode([token_id]))
print(texto_time_machine_tokenizado_subwords)

['The', ' Time', ' Trave', 'ller', ' was', ' exp', 'ounding', ' a', ' rec', 'ond', 'ite', ' matter', ' to', ' us', '.', ' His', ' grey', ' eyes', ' shone', ' and', ' tw', 'ink', 'led', ',', ' and', ' his', ' usually', ' pale', ' face', ' was', ' flushed', ' and', ' animated', '.']


In [60]:
import re
Tokenizando_nivel_palavra_time_machine_text = re.split(r'([,.:;?_!"()\']|--|\s)', time_machine_text)
Tokenizando_nivel_palavra_time_machine_text = [item.strip() for item in Tokenizando_nivel_palavra_time_machine_text if item.strip()]
print(Tokenizando_nivel_palavra_time_machine_text)

['The', 'Time', 'Traveller', 'was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', '.', 'His', 'grey', 'eyes', 'shone', 'and', 'twinkled', ',', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', '.']


Consegui gerar com base no que estudei sobre tokenização 2 vetores:
- Vetor com as "traduções" dos token id para word/subwords de acordo com o vocabulário do gpt-2
- Vetor com as palavras devidamente separadas no texto fornecido de time_machine

Agora, executando um força bruta consigo comparar em ~O(n^2) quais palavras estão presentes no vocabulário do gpt e quais não estão (guardarei estas num vetor).

In [61]:
texto_time_machine_tokenizado_subwords = [item.strip() for item in texto_time_machine_tokenizado_subwords if item.strip()] # precisei tirar uns espaços do tokenizado pelo gpt2 ele mete uns espaços em algumas palavras ;-;
palavras_divididas=[]
for word in Tokenizando_nivel_palavra_time_machine_text:
    flag = 0
    for subword in texto_time_machine_tokenizado_subwords:
        if(word == subword):
            flag = 1
            break
    if(flag==0):
        palavras_divididas.append(word)
print("Palavras que foram divididas em subwords:")
print(palavras_divididas)

Palavras que foram divididas em subwords:
['Traveller', 'expounding', 'recondite', 'twinkled']


Para finalizar (to treinando legal professor desculpa a redundância)

In [62]:
for word in palavras_divididas:
    lista_subwords_traduzidas = []
    lista_subwords_tokenizada_id = tokenizador_tiktoken.encode(f" {word}") # Precisei adicionar um espaço na frente do "Traveller" para ele dividir igual o gpt2 ficando ['Trave', 'ller']. Sem o espaço na frente ele dividia ['T', 'rave', 'ller']. gpt2 lida com palavras com espaço na frente diferente das sem espaço. LEGAL!!
    for subword in lista_subwords_tokenizada_id:
        lista_subwords_traduzidas.append(tokenizador_tiktoken.decode([subword]).strip()) # depois removo o espaço da frente para deixar bonitin
    print(f"A palavra '{word}' foi dividida em -> {lista_subwords_traduzidas}")

A palavra 'Traveller' foi dividida em -> ['Trave', 'ller']
A palavra 'expounding' foi dividida em -> ['exp', 'ounding']
A palavra 'recondite' foi dividida em -> ['rec', 'ond', 'ite']
A palavra 'twinkled' foi dividida em -> ['tw', 'ink', 'led']


## 2 Data loader com diferentes tamanhos de contexto e strides

Durante a aula, vimos como criar um data loader pra treinar uma LLM através da tarefa de prever o próximo token. No caso, o input `x` é uma sequência de tokens e o alvo `y` é o próximo token da sequência `x`. O data loader cria uma janela deslizante que percorre todo o texto, gerando inúmeros exemplos de treino `x, y`. A quantidade de dados de treino gerada pelo data loader vai variar de acordo com o tamanho de `x` (`max_length`) e o tanto que a janela irá deslizar (`stride`) ao longo do texto.

Use o data loader visto durante a aula para tokenizar o texto abaixo com duas configurações distintas:
- `batch_size=4, max_length=2, stride=1`
- `batch_size=4, max_length=6, stride=2`

E responda, quantos exemplos de treino cada configuração o data loader gerou? Lembre-se que cada batch pode conter até 4 exemplos de treino.

In [99]:
time_machine_text = "The Time Traveller (for so it will be convenient to speak of him) \
was expounding a recondite matter to us. His grey eyes shone and \
twinkled, and his usually pale face was flushed and animated. The \
fire burned brightly, and the soft radiance of the incandescent \
lights in the lilies of silver caught the bubbles that flashed and \
passed in our glasses. Our chairs, being his patents, embraced and \
caressed us rather than submitted to be sat upon, and there was that \
luxurious after-dinner atmosphere when thought roams gracefully \
free of the trammels of precision. And he put it to us in this \
way--marking the points with a lean forefinger--as we sat and lazily \
admired his earnestness over this new paradox (as we thought it) \
and his fecundity."

In [100]:
time_text_token_id=tiktoken.get_encoding("gpt2").encode(time_machine_text)
print(time_text_token_id)
print(len(time_text_token_id))

[464, 3862, 43662, 6051, 357, 1640, 523, 340, 481, 307, 11282, 284, 2740, 286, 683, 8, 373, 1033, 9969, 257, 664, 623, 578, 2300, 284, 514, 13, 2399, 13791, 2951, 44193, 290, 665, 676, 992, 11, 290, 465, 3221, 14005, 1986, 373, 44869, 290, 15108, 13, 383, 2046, 11544, 35254, 11, 290, 262, 2705, 2511, 3610, 286, 262, 753, 392, 45470, 7588, 287, 262, 300, 3922, 286, 8465, 4978, 262, 25037, 326, 30050, 290, 3804, 287, 674, 15232, 13, 3954, 18791, 11, 852, 465, 21216, 11, 18079, 290, 1275, 2790, 514, 2138, 621, 8948, 284, 307, 3332, 2402, 11, 290, 612, 373, 326, 35985, 706, 12, 67, 5083, 8137, 618, 1807, 686, 4105, 11542, 2759, 1479, 286, 262, 491, 6475, 1424, 286, 15440, 13, 843, 339, 1234, 340, 284, 514, 287, 428, 835, 438, 4102, 278, 262, 2173, 351, 257, 10904, 1674, 35461, 438, 292, 356, 3332, 290, 37296, 813, 29382, 465, 23176, 1108, 625, 428, 649, 22226, 357, 292, 356, 1807, 340, 8, 290, 465, 27685, 917, 414, 13]
170


In [101]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

# SEU CÓDIGO COM A CLASSE DO DATASET E A FUNÇÃO DO DATA LOADER

class DatasetGPT_aula(Dataset):
    def __init__(self, texto, tokenizador, tamanho_janela, stride):
        self.input_tokens_ids=[]
        self.target_tokens_ids=[]

        dados_tokenizados_id = tokenizador.encode(texto)

        for i in range(0, len(dados_tokenizados_id)-tamanho_janela , stride):
            self.input_tokens_ids.append(torch.tensor(dados_tokenizados_id[i:i+tamanho_janela]))
            self.target_tokens_ids.append(torch.tensor(dados_tokenizados_id[i+1:i+tamanho_janela+1]))

    def __len__(self):
        return len(self.input_tokens_ids)

    def __getitem__(self, idx):
        return self.input_tokens_ids[idx], self.target_tokens_ids[idx]

def criar_data_loader(texto, batch_size=4, tamanho_janela=2, stride=1 ,shuffle=True, drop_last=True, num_workers=0):

    tokenizador_data_load = tiktoken.get_encoding("gpt2")

    dataset = DatasetGPT_aula(texto, tokenizador_data_load, tamanho_janela, stride)

    dataloader = DataLoader(
        dataset, #objeto da classe criada
        batch_size=batch_size, #tamanho do lote. Numero de amostras agrupados por tensor.
        shuffle=shuffle, # embaralhar os dados
        drop_last=drop_last, # se o numero de dados não for perfeitamente divisivel pelo tamanho de lotes ele descarta o restinho (descarta o lote incompleto)
        num_workers=num_workers # processamentos em paralelo processador
    )
    return dataloader

A quantidade de exemplos gerados é definida pela forma como a função `range(start, stop, stride)` do Python funciona no código. 

Temos os seguintes dados iniciais:
* **Total de Tokens:** 170
* **Start (início do laço):** 0
* **Stop (limite final):** `170 - tamanho_janela`

In [102]:
# CHAME O DATA LOADER COM TEXTO ACIMA COM A 1ª CONFIGURAÇÃO
# E CONTE OS EXEMPLOS DE TREINO

data_loader = criar_data_loader(time_machine_text, batch_size=4, tamanho_janela=2, stride=1)

print(data_loader)
print(f"Número de exemplos calculados: {len(data_loader.dataset)}") # meio que os 170 tokens originais - tamanho da janela (2) = 168...
print(f"Número de batches calculados: {len(data_loader)} o que bate com a conta de Exemplos/batch_size: {len(data_loader.dataset)/4}")


Número de exemplos calculados: 168
Número de batches calculados: 42 o que bate com a conta de Exemplos/batch_size: 42.0


O número total de exemplos gerados pelo dataset na 1ª configuração foi de **168**. Contudo, utilizando `drop_last=True`, como a divisão de $168 \div 4$ não é exata ($42$ lotes com sobra de $1$), esse último lote incompleto de apenas 1 amostra é descartado para manter a consistência de tamanho dos lotes. Portanto, o número de exemplos de treino efetivamente utilizados é de **168** (distribuídos em **42 lotes completos** de 4 amostras).

Configuração 1 (Resultado: 168 exemplos)
* **Parâmetros:** `tamanho_janela = 2`, `stride = 1`
* **Cálculo do Stop:** 170 - 2 = 168
* **Como fica o laço:** `range(0, 168, 1)`

**A Conta:**
(168 - 0) / 1 = 168 
O laço andará 168 vezes, gerando exatamente **168 exemplos de treino**. Percorro de 1 em 1 passo a janela então terei de fato 168 elementos.


In [103]:
# CHAME O DATA LOADER COM TEXTO ACIMA COM A 2ª CONFIGURAÇÃO
# E CONTE OS EXEMPLOS DE TREINO

data_loader2 = criar_data_loader(time_machine_text, batch_size=4, tamanho_janela=6, stride=2)

print(data_loader2)
print(f"Número de exemplos calculados: {len(data_loader2.dataset)} na teoria! Como eu deixei o drop_last = True eu descartei esses exemplos que não completaram 1 batch (de 4 exemplos), logo o número correto de exemplos é: 80")
print(f"Número de batches calculados: {len(data_loader2)} o que QUASE bate com a conta de Exemplos/batch_size: {len(data_loader2.dataset)/4}, esse valor de 0.5 (meio batch) foi descartado, logo 2 exemplos de 82 foram descartados.")


Número de exemplos calculados: 82 na teoria! Como eu deixei o drop_last = True eu descartei esses exemplos que não completaram 1 batch (de 4 exemplos), logo o número correto de exemplos é: 80
Número de batches calculados: 20 o que QUASE bate com a conta de Exemplos/batch_size: 20.5, esse valor de 0.5 (meio batch) foi descartado, logo 2 exemplos de 82 foram descartados.


O número total de exemplos gerados pelo dataset foi de **82**. Contudo, seguindo as boas práticas com `drop_last=True`, como a divisão de $82 \div 4$ não é exata ($20$ lotes com sobra de $2$), esse lote incompleto de 2 exemplos é descartado para evitar batches com formatos desiguais. Portanto, o número de exemplos de treino efetivamente utilizados no treinamento é de **80** (distribuídos em **20 lotes completos** de 4 amostras).


## Configuração 2 (Resultado: 82 exemplos)
* **Parâmetros:** `tamanho_janela = 6`, `stride = 2`
* **Cálculo do Stop:** 170 - 6 = 164
* **Como fica o laço:** `range(0, 164, 2)`

**A Conta:**
(164 - 0) / 2 = 82
O laço andará 82 vezes, gerando 82 exemplos, mas pelo drop_last=True gera exatamente **80 exemplos de treino utilizáveis**.